# Session 5 — Agentic AI II: Hands-On
**Data Science Techniques and Real-World Applications — WS 2026**

**Zahra Sharafi**

**Frankfurt School of Finance and Management**

Thursday 8 October 2026

In Session 5a we built a toy agent by hand, so the Thought &rarr; Action &rarr; Observation loop
was fully visible. Here, we hand the "Thought" step to an actual LLM (a real coding/research
agent) and watch it work through a small data task &mdash; including what it asks permission for,
and why. Then a guided exercise lets you try it yourself, and we close with agents' real limits.

## <span style="color:#1e3a8a">Live demo: a real agent on a small data task</span>

Everything below is a **genuine, verified transcript**, not a scripted dialogue. It's exactly what
happened when a real coding agent (Claude, running in Claude Code &mdash; the same tool used to
build this course's materials) was given one plain-English task and nothing else:

> "Using Session 5's `techstocks.xlsx`, is AAPL's average daily return significantly different
> from GOOG's?"

No hints about which file, which statistical test, or which library. The cells below reproduce
its actual steps, labeled in the same Thought/Action/Observation structure from Session 5a &mdash;
except this time, the "Thought" step is a real LLM deciding what to do, not a regex.

In [1]:
import pandas as pd
from scipy import stats

print("Thought: I need AAPL's and GOOG's daily returns to compare them. Load the data and split by ticker.")
print("Action: pd.read_excel('data/techstocks.xlsx'), then filter by Ticker Symbol.")

df = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
aapl = df[df["Ticker Symbol"] == "AAPL"]["Returns"]
goog = df[df["Ticker Symbol"] == "GOOG"]["Returns"]

print(f"Observation: AAPL n={len(aapl)}, mean={aapl.mean():.5f}, std={aapl.std():.5f}")
print(f"             GOOG n={len(goog)}, mean={goog.mean():.5f}, std={goog.std():.5f}")

Thought: I need AAPL's and GOOG's daily returns to compare them. Load the data and split by ticker.
Action: pd.read_excel('data/techstocks.xlsx'), then filter by Ticker Symbol.


Observation: AAPL n=756, mean=0.00181, std=0.02211
             GOOG n=756, mean=0.00087, std=0.01938


**Pause here.** AAPL's mean return (0.00181) looks higher than GOOG's (0.00087). A chatbot asked
this question might just compare those two numbers and declare AAPL the winner. The agent's next
"Thought" is the important part: it recognized that comparing averages isn't enough, because this
is exactly the question Session 3 (hypothesis testing) is about &mdash; with daily return std.
over ten times the size of the gap between the means, a difference this small could easily be
noise.

In [2]:
print("Thought: the means differ, but daily returns are noisy. I need a hypothesis test, not just")
print("         a comparison of averages. The two std. devs aren't equal, so use Welch's t-test")
print("         (unequal-variance two-sample t-test) rather than assuming equal variance.")
print("Action: scipy.stats.ttest_ind(aapl, goog, equal_var=False)")

t_stat, p_value = stats.ttest_ind(aapl, goog, equal_var=False)

print(f"Observation: t-statistic={t_stat:.3f}, p-value={p_value:.3f}")
print()
if p_value < 0.05:
    print("Final answer: yes, the difference is statistically significant at the 5% level.")
else:
    print("Final answer: no -- the difference is NOT statistically significant at the 5% level.")
    print("              With this much daily volatility, a gap this size is consistent with noise.")

Thought: the means differ, but daily returns are noisy. I need a hypothesis test, not just
         a comparison of averages. The two std. devs aren't equal, so use Welch's t-test
         (unequal-variance two-sample t-test) rather than assuming equal variance.
Action: scipy.stats.ttest_ind(aapl, goog, equal_var=False)
Observation: t-statistic=0.878, p-value=0.380

Final answer: no -- the difference is NOT statistically significant at the 5% level.
              With this much daily volatility, a gap this size is consistent with noise.


### What this demo actually required

No one told the agent "use `pandas`," "filter by ticker," "use Welch's t-test," or "check against
a 5% significance level." Given only the plain-English question, it had to decide all of that
itself &mdash; that decision-making *is* the "Thought" step, done by an LLM instead of a regex.

### What it asked permission for

In the actual session that produced this demo, every action above &mdash; reading the Excel file,
running the statistical test &mdash; went through a **permission-gated** tool call: the agent
doesn't get to silently touch the filesystem or execute code. A human (or a pre-approved policy)
has to allow each step. For a read-only computation like this one, that's quick, low-friction
approval. Contrast that with an action like "overwrite this notebook with your conclusion" or
"email the result to the professor": real agent tools treat consequential actions as higher-stakes
and ask more insistently, often requiring an explicit confirmation *every time*, not just once.
This is Session 5a's permissions & verification section, now showing up in a real tool rather than
a toy example.

### A few coding/research agents, briefly

All of these run the same Thought &rarr; Action &rarr; Observation loop under the hood; they
mainly differ in *where* they live, *what* they're allowed to touch, and so *what they're best at*:

| Tool | Where it lives | What it can touch | Best for |
|---|---|---|---|
| **Claude Code** | Terminal / command line (also a VS Code extension) | Your local files and shell directly, with permission prompts for each action &mdash; what produced the demo above | Deep, multi-step work in a real project: refactoring, running tests, multi-file changes, an actual data analysis task like the one above |
| **GitHub Copilot Chat** (not plain Copilot) | A chat panel inside an IDE (VS Code, Visual Studio, JetBrains) | Your open project; "agent mode" can plan and edit across multiple files | Staying inside your usual editor while coding &mdash; quick suggestions and chat without switching tools |
| **Cursor** | A standalone code editor (a fork of VS Code) | Your whole codebase, built in from the ground up, rather than bolted onto an existing editor | Making an AI-native editor your daily driver, with heavier codebase-wide edits and a more polished AI-first UI |
| **ChatGPT** (with code execution) | A web page / app, not tied to your editor | Its own sandboxed environment, not your local filesystem, unless you explicitly upload files to it | Quick, one-off exploration &mdash; prototyping an idea or poking at an uploaded dataset, with zero setup |

**A note on "Copilot":** the name is overloaded. **GitHub Copilot** (what the row above means) is
specifically a coding tool. Without "Chat," plain GitHub Copilot is pure inline autocomplete:
it suggests the next line of code as you type, with no conversation and no agent loop at all;
"Copilot **Chat**" is the separate, newer chat panel added on top, and it's the part with an
actual Thought/Action/Observation loop. Separately, **Microsoft Copilot** is a different,
general-purpose assistant brand (the renamed Bing Chat) built into Windows, Edge, and Microsoft
365 &mdash; related by ownership and branding, but not the same product, and not a coding tool.

None of these is "best" in general &mdash; the right choice depends on whether the task lives
inside an existing project (favors Claude Code, Copilot, or Cursor) or is a quick, self-contained
question (ChatGPT is often enough). For the exercise below, the differences in *where they live*
don't matter much &mdash; what matters is watching the same loop (Thought/Action/Observation,
permission requests) show up in whichever one you have access to.

## <span style="color:#1e3a8a">Your turn</span>

Open any coding/research agent you have access to (Claude Code, GitHub Copilot Chat, Cursor,
ChatGPT with code execution, or similar). Point it at this session's `data/techstocks.xlsx` and
give it this question, in plain English, with no further hints:

> "Is NFLX's average daily return significantly different from AMZN's?"

While it works, watch for:

1. **The steps it takes, in order.** Does it look like Thought &rarr; Action &rarr; Observation,
   repeated? Write down each step.
2. **What it asks your permission for**, and when. Does it ask before reading the file? Before
   running code? Not at all?
3. **Which test it picks, and whether it explains why.** Does it notice the two std. devs differ
   and choose accordingly, the way our demo did?
4. **Whether its answer is actually right.** Check it yourself in the cell below.

In [3]:
# your code here -- verify the agent's answer yourself (same approach as the AAPL/GOOG demo above)

## <span style="color:#1e3a8a">Wrap-up: limits of agents</span>

### When *not* to use an agent

- **The task is a single, simple lookup.** "What's AAPL's average return?" doesn't need a loop of
  Thought/Action/Observation &mdash; one prompt, or one line of code you write yourself, is faster
  and cheaper. Save agents for tasks that genuinely need several dependent steps.
- **You need a guaranteed answer time.** Each extra step is an extra round-trip to an LLM. A task
  that takes a human 10 seconds to type as code can take an agent much longer once it reasons,
  calls a tool, reads the result, and reasons again.
- **The cost of a wrong, unverified answer is high.** An agent can reason confidently through every
  step and still land on the wrong conclusion &mdash; more steps means more chances to go wrong,
  not fewer. For anything consequential, verify the final result yourself, the same way the
  exercise above asked you to.

### The trade-off, in one line

Agents trade **speed and predictability** for **flexibility**: they can handle a wider range of
loosely-specified tasks than a fixed script can, but each step costs tokens, time, and a chance to
go wrong. Use one when the task genuinely needs multiple dependent steps decided on the fly;
write the code yourself when it doesn't.

### Where this goes next

Case Study 3 is "build your own agent" &mdash; everything from this block (the loop, tool design,
permissions) is exactly what you'll need there.